In [1]:
import pandas as pd
from faker import Faker
import random
from typing import OrderedDict
from datetime import timedelta

from faker.providers import BaseProvider

# Add custom classes
class crimes(BaseProvider):
    general_crimes_cat = [
        "Arson",
        "Burglary",
        "Citation",
        "Patrol",
        "Undefined"
    ]

    _crime_relationships = [
        "Suspect",
        "Victim",
        "Witness",
        "Contact",
        "Other"
    ]

    _weighted_crimes = OrderedDict(
    [
        ("Disturbance", 30),
        ("Arson", 10),
        ("Murder", 1),
        ("Routine Patrol", 39),
        ("Other", 20)
    ])

    def general_crime_cat(self):
        return self.random_element(self._general_crimes_cat)
    def crime_relationship(self):
        return self.random_element(self._crime_relationships)
    def weighted_crimes(self, length= None):
        return self.random_elements(self._weighted_crimes, unique=False, use_weighting=True)
    def weighted_crime(self):
        return self.weighted_crimes(1)[0]

class simple_people(BaseProvider):

    _weighted_sex = OrderedDict(
    [
        ("M", 55),
        ("F", 45),
        ("U", 1)
    ])

    _weighted_race = OrderedDict(
    [
        ("W", 57),
        ("B", 40),
        ("A", 2),
        ("U", 1)
    ])

    
    def simple_name(self):
        firstname = fake.first_name()
        lastname = fake.last_name()
        return f"{lastname}; {firstname}"

    def simple_phone(self):
        last_four = ''.join(str(random.randint(0,9)) for _ in range(4))
        return f"512-555-{last_four}"

    def weighted_sex(self, length=None):
        return self.random_elements(self._weighted_sex, unique=False, use_weighting=True)[0]
    
    #def weighted_sex(self):
     #   return self.weighted_sexes(1)[0]

    def weighted_race(self, length=None):
        return self.random_elements(self._weighted_race, unique=False, use_weighting=True)[0]
    
    #def weighted_sex(self):
     #   return self.weighted_sexes(1)[0]



# alterators --- ways to change the values slightly - might want to make these into a class or something

def shift_birthday(date_obj, max_days_shift=10):
    """
    Randomly add or subtract up to max_days_shift days from a date.
    """
    if max_days_shift < 1:
        raise ValueError("max_days_shift must be at least 1.")
    
    # Random shift between -max_days_shift and +max_days_shift
    days_shift = random.randint(-max_days_shift, max_days_shift)
    return date_obj + timedelta(days=days_shift)

def slightly_alter_name(name):
    """
    Slightly alter a given name by changing, swapping, or adding/removing a letter.
    """
    name_chars = list(name)
    
    choice = random.choice(["replace", "swap", "insert", "delete"])
    
    if choice == "replace" and len(name_chars) > 0:
        idx = random.randint(0, len(name_chars) - 1)
        name_chars[idx] = random.choice("abcdefghijklmnopqrstuvwxyz")
    
    elif choice == "swap" and len(name_chars) > 1:
        idx1, idx2 = random.sample(range(len(name_chars)), 2)
        name_chars[idx1], name_chars[idx2] = name_chars[idx2], name_chars[idx1]
    
    elif choice == "insert":
        idx = random.randint(1, len(name_chars))
        name_chars.insert(idx, random.choice("abcdefghijklmnopqrstuvwxyz"))
    
    elif choice == "delete" and len(name_chars) > 1:
        idx = random.randint(0, len(name_chars) - 1)
        del name_chars[idx]
    
    return "".join(name_chars)

    

# Initialize Faker
fake = Faker('en_US')
fake.add_provider(crimes)
fake.add_provider(simple_people)



# Generator Fucntions
def maybe_generate(generator_func, prob=0.5):
    """
    Runs the generator function with a given probability.
    Returns None if skipped.
    """
    if random.random() < prob:
        return generator_func()
    return None


def generate_person():
    """This function generates real-world-like customer data."""

    lic_format = 'TX###{{random_int}}'
    sid_format = '#####{{random_int}}'
    
    
    return {
        "id": fake.uuid4(),
        "name": fake.simple_name(),
        "phone": maybe_generate(fake.simple_phone, prob = 0.5),
        "birthdate": fake.date_of_birth(minimum_age=19, maximum_age=85),
        "sex": fake.weighted_sex(),
        "race": fake.weighted_race(),
        "ssn": maybe_generate(fake.ssn, prob = 0.55), 
        "VLP":  maybe_generate(fake.license_plate, prob = 0.1),
        "OLN": fake.pystr_format(lic_format),
        "SID": fake.pystr_format(sid_format),
        "relationship": fake.crime_relationship(),
        "category": fake.weighted_crime()
        
    }

# creat duplicates

def generate_data_with_duplicates(num_records=10, duplicate_ratio=0.3):
    """
    Generate fake data with intentional duplicates that are slightly altered.
    
    Args:
        num_records (int): Total number of records to generate.
        duplicate_ratio (float): Fraction of records that should be duplicates (0 to 1).
    
    Returns:
        list[dict]: List of fake records.
    """
    if not (0 <= duplicate_ratio <= 1):
        raise ValueError("duplicate_ratio must be between 0 and 1.")
    if num_records <= 0:
        raise ValueError("num_records must be positive.")

    # Step 1: Generate base unique records
    base_records = []
    for _ in range(num_records):
        base_records.append(generate_person())

    # Step 2: Determine how many duplicates to create
    num_duplicates = int(num_records * duplicate_ratio)

    # Step 3: Create duplicates with slight alterations
    duplicates = []
    for _ in range(num_duplicates):
        original = random.choice(base_records)
        altered = original.copy()

        # Slightly alter one or more fields
        if random.random() < 0.5:
            altered["ssn"] = maybe_generate(fake.ssn, prob = 0.20)  # change email
        if random.random() < 0.5:
            altered["phone"] = maybe_generate(fake.simple_phone, prob = 1)  # change phone
        if random.random() < 0.9:
            altered["name"] = slightly_alter_name(altered["name"])
        if random.random() < 0.1:
            altered["birthdate"] = shift_birthday(altered["birthdate"])

        duplicates.append(altered)

    # Step 4: Combine and shuffle
    all_records = base_records + duplicates
    random.shuffle(all_records)

    return all_records


#users_df = pd.DataFrame([generate_person() for _ in range(100)])

users_df = pd.DataFrame(generate_data_with_duplicates(10000))

In [2]:
users_df

,id,name,phone,birthdate,sex,race,ssn,VLP,OLN,SID,relationship,category
0,66d0deef-f8e7-4972-92b6-a806d57096f9,Dean; David,NaN,1973-01-27,M,B,NaN,NaN,TX6074372,032109462,Victim,Routine Patrol
1,b04f366c-1825-409d-ba50-a3a7ec8350bc,Wood; Suszn,NaN,1975-04-09,M,W,NaN,NaN,TX6915379,697625533,Victim,Other
2,f347a8c6-16e7-4241-815a-4b111083176b,Roberts; Kristopher,512-555-4103,1959-08-08,M,B,291-91-0982,NaN,TX4293054,025487524,Victim,Other
3,d02a9ba1-f52e-4018-8dea-b7889f590b68,Brown; Melissa,512-555-8239,1985-08-08,M,W,321-32-6128,NaN,TX0288263,714566988,Contact,Routine Patrol
4,15a20cbb-f563-4997-a501-655c860732a7,Ellis; Joe,NaN,1989-10-06,M,B,NaN,6Q F7973,TX9974728,816095574,Suspect,Arson
...,...,...,...,...,...,...,...,...,...,...,...,...
12995,05181bc9-6901-4c7b-8839-b2c86552dc58,Watson; Angie,512-555-3837,1961-01-17,M,W,080-39-9949,WPV 274,TX6875541,058363701,Witness,Disturbance
12996,a534e76d-5609-46c8-ae67-e136c6627846,Holmes; Justin,512-555-5309,2001-01-09,M,B,NaN,NaN,TX6517820,634644497,Suspect,Routine Patrol
12997,642864c6-7d32-45be-82dd-46c67b01c7a7,Rowe; Kyle,NaN,1999-04-13,M,B,823-44-2408,NaN,TX3259535,283724187,Witness,Routine Patrol
12998,244790a1-9306-4e9e-972c-68be48c95fa8,Kelly; Amy,512-555-3732,1946-11-11,M,W,NaN,NaN,TX202835,174151365,Other,Disturbance


In [77]:
users_df.reset_index(inplace = True)

In [78]:
from splink import splink_datasets
from splink import DuckDBAPI

df = users_df.copy()
db_api = DuckDBAPI()
df_sdf = db_api.register(df)
df_sdf.as_duckdbpyrelation().limit(5).show(max_width=10000)

┌───────┬──────────────────────────────────────┬────────────────────┬──────────────┬────────────┬─────────┬─────────┬─────────────┬─────────┬───────────┬───────────┬─────────────┬────────────┬──────────────┬────────────────┐
│ index │                  id                  │        name        │    phone     │ birthdate  │   sex   │  race   │     ssn     │   VLP   │    OLN    │    SID    │  latitude   │ longitude  │ relationship │    category    │
│ int64 │               varchar                │      varchar       │   varchar    │    date    │ varchar │ varchar │   varchar   │ varchar │  varchar  │  varchar  │   double    │   double   │   varchar    │    varchar     │
├───────┼──────────────────────────────────────┼────────────────────┼──────────────┼────────────┼─────────┼─────────┼─────────────┼─────────┼───────────┼───────────┼─────────────┼────────────┼──────────────┼────────────────┤
│     0 │ e080bb2d-6b5d-4dc9-b361-3c337aee8971 │ Roberts; Jennifer  │ NULL         │ 2005-07-28 │ M 

In [79]:
from splink import DuckDBAPI, block_on
from splink.blocking_analysis import (
    chart_comparisons_from_blocking_rules,
)


chart_comparisons_from_blocking_rules(
    df_sdf,
    blocking_rules=[
        block_on("name"),
        block_on("birthdate", "race"),
        block_on("phone", "ssn"),
        block_on("name", "birthdate", "sex"),
        block_on("OLN", "SID"),
        block_on("name", "birthdate")
    ],
    link_type="dedupe_only",
    unique_id_column_name = 'index',
    record_sample_proportion=1,
)

SplinkChart (use .to_html() or .save() to export HTML)

In [86]:
from splink import Linker, SettingsCreator

settings = SettingsCreator(
    link_type="dedupe_only",
    blocking_rules_to_generate_predictions=[
        block_on("name"),
        block_on("birthdate", "race"),
        block_on("phone", "ssn"),
        block_on("name", "birthdate", "sex"),
        block_on("OLN", "SID")
    ],
    retain_intermediate_calculation_columns=True,
    unique_id_column_name = 'index',
)

linker = Linker(df_sdf, settings)

In [87]:
df_predict = linker.inference.deterministic_link()
df_predict.as_duckdbpyrelation().limit(10).show(max_width=10000)

Blocking time: 0.07 seconds


┌─────────┬─────────┬──────────────┬──────────────┬───────────┬───────────┬─────────┬─────────┬───────────┬───────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────┬─────────┬────────────────────┬─────────────────────┬───────────┐
│ index_l │ index_r │   phone_l    │   phone_r    │   OLN_l   │   OLN_r   │ race_l  │ race_r  │   SID_l   │   SID_r   │ birthdate_l │ birthdate_r │    ssn_l    │    ssn_r    │  sex_l  │  sex_r  │       name_l       │       name_r        │ match_key │
│  int64  │  int64  │   varchar    │   varchar    │  varchar  │  varchar  │ varchar │ varchar │  varchar  │  varchar  │    date     │    date     │   varchar   │   varchar   │ varchar │ varchar │      varchar       │       varchar       │  varchar  │
├─────────┼─────────┼──────────────┼──────────────┼───────────┼───────────┼─────────┼─────────┼───────────┼───────────┼─────────────┼─────────────┼─────────────┼─────────────┼─────────┼─────────┼────────────────────┼─────────────────────┼─────────

In [82]:
clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    df_predict
)

Completed iteration 1, num edges remaining to process: 0
